일단 기본 4개 노드 4개 원형 엣지 구성으로 감


In [ ]:
import torch
import numpy as np
import quimb as qu
n=10

In [ ]:
def analytic(N, edge_full_data, node_full_data, num):
  analytical_energies = []  # analytical result
  analytical_states = []  # analytical result

  for k in range(num):
    H = np.zeros((2**N, 2**N), dtype=complex)  # numpy 영행렬

    for (i, j), (J_xx, J_yy, J_zz) in edge_full_data[k].items():
      # print(q1, q2)
      # print(J_xx.item(), J_yy, J_zz)
      H += J_xx.item() * (
          qu.ikron(qu.pauli('X'), dims=[2]*N, inds=[i]) @
          qu.ikron(qu.pauli('X'), dims=[2]*N, inds=[j])
      )
      # YY 항
      H += J_yy.item() * (
          qu.ikron(qu.pauli('Y'), dims=[2]*N, inds=[i]) @
          qu.ikron(qu.pauli('Y'), dims=[2]*N, inds=[j])
      )
      # ZZ 항
      H += J_zz.item() * (
          qu.ikron(qu.pauli('Z'), dims=[2]*N, inds=[i]) @
          qu.ikron(qu.pauli('Z'), dims=[2]*N, inds=[j])
      )

    K_x = node_full_data[k][0][-1].item()
    for q in range(N):
      H += K_x*qu.ikron(qu.pauli('X'), dims=[2]*N, inds=[q])
    if k%100 == 0:
      print(k)

    analytical_energies.append(qu.linalg.base_linalg.groundenergy(H))
    analytical_states.append(qu.linalg.base_linalg.groundstate(H).reshape(-1))

  return analytical_energies, analytical_states

def mse(vector1, vector2):
    return np.mean((np.array(vector1) - np.array(vector2)) ** 2)


다른 범위 새로운 데이터

In [ ]:
test_edge_full_data = torch.load('edge_full_data_test.pt')
test_node_full_data = torch.load('node_full_data_test.pt')


In [ ]:
len(test_edge_full_data)

In [ ]:
analytical_energies_test, analytical_states_test =  analytic(n,test_edge_full_data, test_node_full_data,len(test_edge_full_data))
print("analytical_energies_test = ",analytical_energies_test)

In [ ]:
torch.save(analytical_energies_test, 'analytical_energies_test.pt')
torch.save(analytical_states_test, 'analytical_states_test.pt')

